# Lab 4: 上下文与内存层 — Context & Memory

对应 Harness 六层架构：第 ❹ 层（教学顺序调整：安全层之后学记忆）

## 学习目标

- 理解 **context window** 是 agent 的"工作记忆"，需要主动管理
- 实现 **micro-compaction**：对话内自动压缩旧的工具输出
- 实现 **project memory**：自动发现并加载 CLAUDE.md（项目级记忆）

## 痛点回顾

Lab 3 加了安全层后 agent 安全了，但有一个致命问题：
1. **长对话 token 爆炸** — 大量工具调用结果撑满 context window
2. **关掉即失忆** — 重启后一切从零开始

**本 Lab 的目标：给 agent 装上"记忆系统"。**

## 对应 Claude Code 源码

- `compact/apiMicrocompact.ts` — 上下文微压缩
- `memdir/` — 项目记忆系统（CLAUDE.md 发现与加载）

本 Lab 覆盖的记忆机制：

| 层级 | 作用 | 时间尺度 |
|------|------|----------|
| Micro-compaction | 对话内压缩旧消息 | 当前对话 |
| Project memory | 项目级知识（CLAUDE.md） | 跨项目 |

> 注：Session persistence（会话持久化）将在 Lab 6（状态与持久层）中实现。


---
## 第一步：环境准备 + 回顾 Lab 2-3 代码

我们需要 Lab 2 的工具定义和 Lab 3 的权限系统。


In [ ]:
# === 环境准备 + Lab 2 核心代码回顾 ===
from anthropic import AnthropicBedrock
import  os
from dataclasses import dataclass
from typing import Literal

client = AnthropicBedrock(aws_region="us-west-2")
MODEL = "global.anthropic.claude-sonnet-4-6"

# --- 三个核心工具（来自 Lab 2)---
from utils.tools import WriteFileTool,ReadFileTool,BashTool,Tool

# --- 权限系统（预览，完整版见 Lab 3）---
from utils.permissions import SmartPermissionChecker
DANGEROUS_PATTERNS = ["rm ", "sudo ", "chmod ", "chown ", "mkfs", "dd if=", "> /etc/"]
SAFE_PATTERNS = ["ls", "cat ", "head ", "tail ", "echo ", "pwd", "python ", "python3 ",
                 "git status", "git log", "git diff", "wc ", "grep ", "find ", "which "]

TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]
checker = SmartPermissionChecker()
print(f"✓ Lab 2 代码回顾完成，{len(TOOLS)} 个工具 + 权限检查器就绪")


✓ Lab 2 代码回顾完成，3 个工具 + 权限检查器就绪


---
## 第二步：实现 Micro-Compaction（上下文压缩）

### 为什么需要压缩？

每次工具调用都会产生 `tool_result`，比如 `bash ls -la` 可能返回上百行输出。
多轮对话后，这些历史结果会撑满 context window（通常 100K-200K tokens）。

### Claude Code 的策略：Micro-Compaction

- 监控 token 使用量
- 当接近阈值时，将**旧的 tool_result** 压缩为摘要
- 保留最近的完整对话
- 插入 `SystemCompactBoundaryMessage` 标记压缩点

我们实现一个简化版本。

In [4]:
# Micro-Compaction 上下文压缩器（对应 Claude Code 的 apiMicrocompact.ts）

class ContextManager:
    """管理对话上下文，防止 token 溢出。"""
    
    def __init__(self, max_tokens: int = 50000):
        self.max_tokens = max_tokens
    
    def estimate_tokens(self, text: str) -> int:
        """粗略估算 token 数（1 token ≈ 3 个字符）"""
        return len(str(text)) // 3
    
    def count_message_tokens(self, messages: list) -> int:
        """计算整个消息列表的 token 估算"""
        return sum(self.estimate_tokens(str(m)) for m in messages)
    
    def should_compact(self, messages: list) -> bool:
        """是否需要压缩？当 token 用量超过阈值的 80% 时触发"""
        total = self.count_message_tokens(messages)
        return total > self.max_tokens * 0.8
    
    def compact(self, messages: list) -> list:
        """压缩旧消息中的 tool_result。
        
        策略：
        1. 保留最近 N 轮完整对话
        2. 将旧的 tool_result 内容压缩为摘要
        3. 插入压缩边界标记
        """
        if len(messages) <= 8:
            return messages  # 太短，不需要压缩
        
        # 保留最近 8 条消息（约 4 轮对话）
        keep_recent = 8
        old_messages = messages[:-keep_recent]
        recent_messages = messages[-keep_recent:]
        
        # 压缩旧消息中的 tool_result
        compacted = []
        for msg in old_messages:
            if isinstance(msg.get("content"), list):
                new_content = []
                for block in msg["content"]:
                    if isinstance(block, dict) and block.get("type") == "tool_result":
                        original = str(block.get("content", ""))
                        if len(original) > 200:
                            # 压缩：只保留前 100 字符 + 长度信息
                            block = {
                                **block,
                                "content": f"[已压缩] {original[:100]}... (原始 {len(original)} 字符)",
                            }
                    new_content.append(block)
                msg = {**msg, "content": new_content}
            compacted.append(msg)
        
        # 插入压缩边界标记
        boundary = {
            "role": "user",
            "content": "[系统提示：较早的对话上下文已被压缩以节省 token。最近的对话保持完整。]"
        }
        
        return compacted + [boundary] + recent_messages

# 测试
ctx = ContextManager(max_tokens=1000)  # 用小值方便测试
test_msgs = [{"role": "user", "content": "x" * 500} for _ in range(5)]
print(f"压缩前 token 估算: {ctx.count_message_tokens(test_msgs)}")
print(f"需要压缩？{ctx.should_compact(test_msgs)}")

压缩前 token 估算: 885
需要压缩？True


---
## 第三步：实现 Project Memory（项目记忆）

### CLAUDE.md 模式

Claude Code 会自动搜索当前目录及父目录中的 `CLAUDE.md` 文件，
将其内容注入 system prompt，作为项目级的持久记忆。

这样 agent 每次启动就自动了解：
- 项目的编码规范
- 特殊的架构约定
- 需要注意的事项

**这是跨项目、跨会话的记忆——不依赖对话历史。**

In [5]:
# Project Memory 项目记忆（对应 Claude Code 的 memdir/ 系统）
from utils.project_memory import ProjectMemory

# 测试：发现当前目录的 CLAUDE.md
project_memory = ProjectMemory()
print("搜索项目记忆文件...")
found = project_memory.discover(os.getcwd())
if found:
    print(f"\n内容预览:\n{found[:300]}")
else:
    print("  未找到项目记忆文件。")
    print("  提示：在当前目录创建 CLAUDE.md 文件后重新运行此 cell。")

搜索项目记忆文件...

内容预览:
# 项目记忆: /home/ubuntu/workspace/agent_harness/labs/CLAUDE.md
# 项目记忆示例

这是一个 Agent Harness Workshop 的实验项目。

## 编码规范
- 所有回答请使用中文
- 代码注释使用中文
- 文件名使用英文小写+下划线

## 项目说明
这个目录包含 6 个递进式实验 notebook，从裸 agent 循环逐步构建一个完整的 mini harness。

# 项目记忆: /home/ubuntu/AGENTS.md
---
summary: "Workspace template for AGENTS.md


---
## 第四步：整合到 Agent Loop

现在把记忆系统整合到 agent loop 中：

```
启动时:
  → ProjectMemory.discover() → 注入 system prompt

每轮对话后:
  → ContextManager.should_compact()? → compact()
```


In [9]:
def run_agent_loop(system_prompt, tools_list, user_messages, max_turns=20, permission_checker=None, context_manager=None):
    """Agent Loop（notebook 友好版：消息列表驱动，无 input()）"""
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        if context_manager and context_manager.should_compact(messages):
            before = context_manager.count_message_tokens(messages)
            messages = context_manager.compact(messages)
            after = context_manager.count_message_tokens(messages)
            print(f"  上下文已压缩: {before} -> {after} tokens")

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    elif not tool.validate(blk.input):
                        result, is_error = "参数校验失败", True
                    else:
                        perm = permission_checker.check(blk.name, blk.input) if permission_checker else "allow"
                        if perm == "deny":
                            result, is_error = "权限拒绝", True
                            print(f"  拒绝 {blk.name}")
                        elif perm == "ask":
                            confirm = input("     允许执行？(y/n): ")
                            if confirm.strip().lower() in ("y", "yes"):
                                result, is_error = tool.execute(blk.input), False
                                print(f"  [✅ 已执行] {blk.name}")

                            else:
                                result, is_error = "用户拒绝了此操作。", True
                                print(f"     ❌ 用户拒绝")
                        else:
                            result, is_error = tool.execute(blk.input), False
                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {result[:300]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    print("--- Agent Loop 结束 ---")
    return last_text

print("run_agent_loop 就绪")


run_agent_loop 就绪


---
## 第五步：验证运行

### 测试：项目记忆 + 上下文压缩

运行 agent，注意观察：
- system prompt 中是否包含了 CLAUDE.md 的内容
- 多轮对话后是否触发了上下文压缩

In [10]:
# 运行带记忆系统的 Agent

BASE_PROMPT = """你是一个编程助手，可以读写文件和执行 bash 命令。
请用中文回答。当需要操作文件或执行命令时，使用提供的工具。"""

pm = ProjectMemory()
system_prompt = pm.build_system_prompt(BASE_PROMPT, os.getcwd())
ctx_mgr = ContextManager(max_tokens=50000)

print("=" * 60)
print("Lab 3: Agent Loop + 工具 + 上下文管理")
print("=" * 60)

run_agent_loop(
    system_prompt=system_prompt,
    tools_list=TOOLS,
    user_messages=[
        "列出当前目录的文件",
        "读取 CLAUDE.md 的内容",
        "用 bash 查看当前 Python 版本",
        "总结一下你刚才做了什么",
    ],
    permission_checker=checker,
    context_manager=ctx_mgr,
)


Lab 3: Agent Loop + 工具 + 上下文管理

[轮次 1] 用户: 列出当前目录的文件


  [bash] {'command': 'ls -la'}
   -> total 280
drwxrwxr-x 4 ubuntu ubuntu  4096 Apr  3 01:15 .
drwxrwxr-x 9 ubuntu ubuntu  4096 Apr  2 08:39 ..
drwxrwxr-x 2 ubuntu ubuntu  4096 Apr  2 03:43 .sessions
-rw-rw-r-- 1 ubuntu ubuntu   322 Apr  2 06:53 CLAUDE.md
-rw-rw-r-- 1 ubuntu ubuntu    17 Apr  2 09:11 audit_test.py
-rw-rw-r-- 1 ubuntu u

Assistant: 当前目录下共有以下文件和文件夹：

| 名称 | 类型 | 大小 | 修改时间 |
|------|------|------|----------|
| `.sessions/` | 目录 | - | Apr 2 03:43 |
| `CLAUDE.md` | 文件 | 322 B | Apr 2 06:53 |
| `audit_test.py` | 文件 | 17 B | Apr 2 09:11 |
| `lab1_prompt_guidance.ipynb` | Notebook | 75 KB | Apr 2 08:22 |
| `lab2_tool_execution.ipynb` | Notebook | 45 KB | Apr 2 08:53 |
| `lab3_safety_permission.ipynb` | Notebook | 28 KB | Apr 2 09:10 |
| `lab4_context_memory.ipynb` | Notebook | 32 KB | Apr 3 01:19 |
| `lab5_planning_coordination.i

[轮次 2] 用户: 读取 CLAUDE.md 的内容
  [read_file] {'path': 'CLAUDE.md'}
   -> # 项目记忆示例

这是一个 Agent Harness Workshop 的实验项目。

## 编码规范
- 所有回答请使用中文
- 代码注释使用中文
-

'好的，以下是我刚才所做操作的总结：\n\n1. **列出当前目录文件** (`ls -la`)\n   - 查看了当前目录的所有文件和文件夹\n   - 发现了 6 个实验 Notebook（lab1~lab6）、工具目录 `utils/`、配置文件 `CLAUDE.md` 等\n\n2. **读取 `CLAUDE.md` 文件内容** (`read_file`)\n   - 了解了项目的编码规范：使用中文回答、中文注释、英文小写+下划线命名\n   - 确认这是一个 Agent Harness Workshop 实验项目，包含 6 个递进式实验 Notebook\n\n3. **查看 Python 版本** (`python3 --version`)\n   - 确认当前环境使用的是 **Python 3.12.3**\n\n---\n\n总体来说，这三步操作是对当前工作环境的一次基本**环境探索与确认**，了解了目录结构、项目规范和运行环境。'

---
## 完整集成示例：上下文管理 + 安全 + 工具 + 提示引导

集成前四层 Harness（按教学顺序）：
- ❶ 提示与引导层（CLAUDE.md + Hooks）
- ❷ 工具与执行层（Tool 接口 + 执行）
- ❸ 验证与安全层（权限检查 + Bash 分类器）← Lab 3
- ❹ 上下文与内存层（Compaction + ProjectMemory）← 本 Lab


In [11]:
# === 完整集成：❶提示 + ❷工具 + ❸安全 + ❹上下文 ===
# 从 utils/ 导入前面 Lab 的实现

from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner
from utils.permissions import SmartPermissionChecker

# ❶ 项目记忆
pm = ProjectMemory()
full_prompt = pm.build_system_prompt('你是编程助手。用中文回答。', os.getcwd())

# ❶ Hooks: 工具调用计数 + 安全审计
tool_call_count = {'total': 0}
audit_trail = []

def counting_and_audit_hook(name, inp):
    from datetime import datetime
    tool_call_count['total'] += 1
    perm = perm_checker.check(name, inp)
    audit_trail.append({'tool': name, 'permission': perm, 'time': datetime.now().isoformat()})
    print(f"  [Hook #{tool_call_count['total']}] {name} -> 权限={perm}")
    return inp

hooks = HookRunner()
hooks.register_pre_tool(counting_and_audit_hook)

# ❸ 权限检查
perm_checker = SmartPermissionChecker()

# ❹ 上下文管理
ctx = ContextManager(max_tokens=50000)

# 运行
print('=' * 60)
print('完整集成：❶提示 + ❷工具 + ❸安全 + ❹上下文')
print('=' * 60)

run_agent_loop(
    system_prompt=full_prompt,
    tools_list=TOOLS,
    user_messages=[
        '列出当前目录的文件',                # bash ls -> allow
        '读取 CLAUDE.md',                     # read_file -> allow
        '创建 test_integration.py 写入 print(1)',  # write -> ask(自动放行)
        '运行 python test_integration.py',    # python -> allow
        '删除 test_integration.py',           # rm -> ask(自动放行)
        '总结你刚才做了什么',
    ],
    permission_checker=perm_checker,
    context_manager=ctx,
)

# 汇总
print(f'\n--- 统计 ---')
print(f'工具调用总计: {tool_call_count["total"]} 次')
print(f'\n--- 安全审计记录 ({len(audit_trail)} 条) ---')
for entry in audit_trail:
    print(f"  [{entry['permission']:5s}] {entry['tool']} @ {entry['time']}")


完整集成：❶提示 + ❷工具 + ❸安全 + ❹上下文

[轮次 1] 用户: 列出当前目录的文件


  [bash] {'command': 'ls -la'}
   -> total 280
drwxrwxr-x 4 ubuntu ubuntu  4096 Apr  3 01:15 .
drwxrwxr-x 9 ubuntu ubuntu  4096 Apr  2 08:39 ..
drwxrwxr-x 2 ubuntu ubuntu  4096 Apr  2 03:43 .sessions
-rw-rw-r-- 1 ubuntu ubuntu   322 Apr  2 06:53 CLAUDE.md
-rw-rw-r-- 1 ubuntu ubuntu    17 Apr  2 09:11 audit_test.py
-rw-rw-r-- 1 ubuntu u

Assistant: 当前目录下共有以下文件和文件夹：

| 类型 | 名称 | 大小 | 修改时间 |
|------|------|------|----------|
| 📁 目录 | `.sessions` | - | Apr 2 03:43 |
| 📁 目录 | `utils` | - | Apr 2 08:21 |
| 📄 文件 | `CLAUDE.md` | 322 B | Apr 2 06:53 |
| 📄 文件 | `audit_test.py` | 17 B | Apr 2 09:11 |
| 📄 文件 | `lab1_prompt_guidance.ipynb` | 75 KB | Apr 2 08:22 |
| 📄 文件 | `lab2_tool_execution.ipynb` | 45 KB | Apr 2 08:53 |
| 📄 文件 | `lab3_safety_permission.ipynb` | 29 KB | Apr 2 09:10 |
| 📄 文件 | `lab4_context_memory.ipynb` | 32 KB | Apr 3 01:19 |
| 📄 文

[轮次 2] 用户: 读取 CLAUDE.md
  [read_file] {'path': 'CLAUDE.md'}
   -> # 项目记忆示例

这是一个 Agent Harness Workshop 的实验项目。

## 编码规范
- 所有回答请使用中文
- 代码注释使用中文
- 文件名

---
## 源码对照：Claude Code 的三层记忆

```
Layer 1: Micro-compaction（apiMicrocompact.ts）
  → 对话内自动压缩
  → 折叠旧的 tool_result 为摘要
  → 插入 SystemCompactBoundaryMessage 标记
  → 触发条件：token 用量接近 context window 上限

Layer 2: Session persistence（history.ts）  ← Lab 6 实现
  → 保存到 ~/.claude/sessions/{uuid}.json
  → 包含 messages + metadata + token 消耗 + 费用
  → /resume 命令恢复最近会话
  → /rewind 命令回退到之前的状态

Layer 3: Project memory（memdir/）
  → 层级发现: 当前目录 → 父目录 → home 目录
  → 文件: CLAUDE.md, .claude/CLAUDE.md
  → 自动注入 system prompt
  → 还有 auto-memory: 自动保存用户偏好到 ~/.claude/memory/
```

---
## 小结

### 本 Lab 你学到了什么

1. **Context window 是有限的工作记忆** — 必须主动管理
2. **Micro-compaction** — 压缩旧的工具输出，保留最近上下文
3. **Project memory (CLAUDE.md)** — 跨项目、跨会话的持久知识

### 痛点预告

现在 agent 能做事了、上下文也管理好了，但它仍然**复杂任务一个 agent 忙不过来**——对 agent 说“删除所有文件”，它会毫不犹豫地执行。

→ **下一个 Lab：Lab 5 规划与协调层——给 agent 装上多 Agent 协作能力。**
